In [2]:
"""
Auto-fill manual review truth/evidence fields from local config/test files.
This version stores ONLY file names (no directories) in:
- yaml_paths
- build_paths
- androidtest_paths
"""

import os
import re
import csv
import glob
import pandas as pd
from pathlib import Path

# -------------------- CONFIG: EDIT IF NEEDED --------------------
MANUAL_SHEET = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Manual_Review_Sheet__Prefilled_RQ1.csv"
CONFIG_DIR   = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
TEST_DIR     = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Test_Files"
OUTPUT_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\manual_review_sheet_autofilled.csv"

SNIPPET_CHARS = 160

CONFIG_PATTERNS = [
    "**/*.yml", "**/*.yaml",
    "**/*.json",
    "**/*.sh", "**/*.bash", "**/*.cmd", "**/*.bat",
    "**/*.gradle", "**/*.gradle.kts", "**/settings.gradle*", "**/build.gradle*",
    "**/*.txt"
]

TEST_PATTERNS = [
    "**/*.kt", "**/*.java", "**/*.groovy", "**/*.xml", "**/*.md", "**/*.txt"
]

# -------------------- DETECTION REGEXES --------------------
RE_CI_TEST_CMDS = re.compile(
    r"(gradle[w]?\S*\s+(?:[:\w\-]+:)?connected(?:Android)?Test\b|"
    r"gradle[w]?\S*\s+connectedCheck\b|"
    r"gcloud\s+firebase\s+test\s+android\s+run\b|"
    r"firebase\s+test\s+android\s+run\b|"
    r"browserstack|bstack|saucectl|sauce\s+ctl|bitrise.*(virtual|device).*(test|testing))",
    re.IGNORECASE
)
RE_DEVICE_SETUP = re.compile(
    r"(sdkmanager\b|avdmanager\b|emulator\b|adb\s+start-server\b|managedDevices\s*\{|"
    r"actions/setup-android|install-android-sdk|avd\s+create)",
    re.IGNORECASE
)
RE_VENDOR = re.compile(
    r"(firebase\s+test\s+lab|gcloud\s+firebase\s+test|browserstack|bstack|sauce\s*labs|bitrise)",
    re.IGNORECASE
)
RE_BUILD_ANCHORS = re.compile(
    r"(testInstrumentationRunner|androidTestImplementation|androidx\.test|espresso|uiautomator|"
    r"testOptions\s*\{\s*managedDevices|connectedAndroidTest|connectedCheck|"
    r"com\.google\.firebase\.testlab|devicefarm|browserstack)",
    re.IGNORECASE | re.DOTALL
)
RE_TEST_INSTRUMENTATION = re.compile(
    r"(AndroidJUnit4|InstrumentationRegistry|androidx\.test\.ext|androidx\.test\.espresso|@RunWith\s*\(\s*AndroidJUnit4)",
    re.IGNORECASE
)
RE_PATH_ANDROIDTEST = re.compile(r"androidtest", re.IGNORECASE)

# -------------------- HELPERS --------------------
def safe_read_text(path: Path, max_mb: float = 5.0) -> str:
    try:
        if path.stat().st_size > max_mb * 1024 * 1024:
            return ""
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception:
        return ""

def normalize_repo_key(s: str) -> str:
    if not isinstance(s, str) or not s:
        return ""
    ss = s.strip().lower()
    if ss.startswith("http"):
        parts = [p for p in ss.split("/") if p]
        if len(parts) >= 2:
            owner = parts[-2]
            repo = parts[-1].replace(".git", "")
            ss = f"{owner}/{repo}"
    ss = ss.replace("\\", "/")
    if "/" in ss:
        owner, repo = ss.split("/")[-2], ss.split("/")[-1]
        base = f"{owner}_{repo}"
    else:
        base = ss
    base = re.sub(r"[^a-z0-9_]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_")
    return base

def file_prefix_before_dunder(path: Path) -> str:
    """Part before first '__' (no extension)."""
    return path.stem.split("__", 1)[0].lower()

def collect_repo_files(base_dir: str, patterns):
    mapping = {}
    base = Path(base_dir)
    for pat in patterns:
        for p in base.glob(pat):
            if not p.is_file():
                continue
            pref_norm = normalize_repo_key(file_prefix_before_dunder(p))
            mapping.setdefault(pref_norm, []).append(p)
    return mapping

def find_matches_for_repo(repo_key: str, mapping: dict):
    return mapping.get(repo_key, [])

def any_match(regex: re.Pattern, text: str):
    m = regex.search(text)
    if not m:
        return False, ""
    i = m.start()
    start = max(0, i - SNIPPET_CHARS//2)
    end = min(len(text), i + SNIPPET_CHARS//2)
    snippet = text[start:end].replace("\n", " ").replace("\r", " ").strip()
    return True, snippet

def merge_field(old, new_items, sep="; "):
    old = "" if pd.isna(old) else str(old)
    # unique, preserve order
    new = sep.join(dict.fromkeys(new_items))
    return new if not old else old + sep + new

# -------------------- MAIN --------------------
def main():
    df = pd.read_csv(MANUAL_SHEET)

    needed = [
        "html_url","full_name","provider","stratum","YAML_pred","Build_pred","AT_pred","group_pred",
        "yaml_signal_true","build_true","at_true","ci_runs_true","group_true",
        "reason_codes","reason_notes","yaml_paths","build_paths","androidtest_paths",
        "ci_command_evidence","device_setup_evidence","vendor_service_evidence",
        "reviewer_initial","reviewer_secondary","resolution","resolution_notes","review_date"
    ]
    for c in needed:
        if c not in df.columns:
            df[c] = ""

    print("Indexing config files...")
    cfg_map = collect_repo_files(CONFIG_DIR, CONFIG_PATTERNS)
    print("Indexing test files...")
    test_map = collect_repo_files(TEST_DIR, TEST_PATTERNS)

    rows = []
    auto_counts = dict(yaml=0, build=0, at=0, ci=0)

    for _, r in df.iterrows():
        html_url = str(r.get("html_url", "") or "")
        full_name = str(r.get("full_name", "") or "")
        repo_key = normalize_repo_key(full_name) or normalize_repo_key(html_url)

        cfg_files = find_matches_for_repo(repo_key, cfg_map)
        tst_files = find_matches_for_repo(repo_key, test_map)

        yaml_true = 0
        build_true = 0
        at_true = 0
        ci_true = 0

        # *** store FILE NAMES (no directories) ***
        yaml_files = []
        build_files = []
        at_files = []

        ci_cmd_snips = []
        dev_setup_snips = []
        vendor_snips = []

        # Scan config files
        for p in cfg_files:
            text = safe_read_text(p)
            if not text:
                continue
            fname = p.name  # <-- file name only

            matched, snip = any_match(RE_CI_TEST_CMDS, text)
            if matched:
                yaml_true = 1
                yaml_files.append(fname)
                ci_cmd_snips.append(snip)

            matched, snip = any_match(RE_DEVICE_SETUP, text)
            if matched:
                if fname not in yaml_files:
                    yaml_files.append(fname)
                dev_setup_snips.append(snip)

            matched, snip = any_match(RE_VENDOR, text)
            if matched:
                if fname not in yaml_files:
                    yaml_files.append(fname)
                vendor_snips.append(snip)

            matched, snip = any_match(RE_BUILD_ANCHORS, text)
            if matched:
                build_true = 1
                build_files.append(fname)

        # Explicit build files (gradle)
        for p in cfg_files:
            name = p.name.lower()
            if name.startswith(("build.gradle", "settings.gradle")) or name.endswith((".gradle", ".gradle.kts")):
                text = safe_read_text(p)
                if not text:
                    continue
                fname = p.name
                matched, snip = any_match(RE_BUILD_ANCHORS, text)
                if matched:
                    build_true = 1
                    if fname not in build_files:
                        build_files.append(fname)

        # Scan test files
        for p in tst_files:
            fname = p.name
            full = str(p).lower()
            if RE_PATH_ANDROIDTEST.search(full):
                at_files.append(fname)
            text = safe_read_text(p)
            if not text:
                continue
            matched, snip = any_match(RE_TEST_INSTRUMENTATION, text)
            if matched and fname not in at_files:
                at_files.append(fname)

        at_true = 1 if any(f.lower().endswith((".kt", ".java")) for f in at_files) else (1 if len(at_files) > 0 else 0)
        ci_true = 1 if (len(ci_cmd_snips) > 0 and len(dev_setup_snips) > 0) else 0

        auto_counts["yaml"]  += yaml_true
        auto_counts["build"] += build_true
        auto_counts["at"]    += at_true
        auto_counts["ci"]    += ci_true

        out = dict(r)
        if pd.isna(out.get("yaml_signal_true")) or str(out.get("yaml_signal_true")).strip()=="":
            out["yaml_signal_true"] = yaml_true
        if pd.isna(out.get("build_true")) or str(out.get("build_true")).strip()=="":
            out["build_true"] = build_true
        if pd.isna(out.get("at_true")) or str(out.get("at_true")).strip()=="":
            out["at_true"] = at_true
        if pd.isna(out.get("ci_runs_true")) or str(out.get("ci_runs_true")).strip()=="":
            out["ci_runs_true"] = ci_true

        # *** write FILE NAMES into *_paths columns ***
        out["yaml_paths"] = merge_field(out.get("yaml_paths", ""), yaml_files)
        out["build_paths"] = merge_field(out.get("build_paths", ""), build_files)
        out["androidtest_paths"] = merge_field(out.get("androidtest_paths", ""), at_files)

        out["ci_command_evidence"] = merge_field(out.get("ci_command_evidence", ""), ci_cmd_snips, sep=" | ")
        out["device_setup_evidence"] = merge_field(out.get("device_setup_evidence", ""), dev_setup_snips, sep=" | ")
        out["vendor_service_evidence"] = merge_field(out.get("vendor_service_evidence", ""), vendor_snips, sep=" | ")

        rows.append(out)

    out_df = pd.DataFrame(rows)

    cols = [
        "html_url","full_name","provider","stratum","YAML_pred","Build_pred","AT_pred","group_pred",
        "yaml_signal_true","build_true","at_true","ci_runs_true","group_true",
        "reason_codes","reason_notes","yaml_paths","build_paths","androidtest_paths",
        "ci_command_evidence","device_setup_evidence","vendor_service_evidence",
        "reviewer_initial","reviewer_secondary","resolution","resolution_notes","review_date"
    ]
    for c in cols:
        if c not in out_df.columns:
            out_df[c] = ""
    out_df = out_df[cols]

    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"Done. Wrote: {OUTPUT_CSV}")
    print("Auto-detected counts (1's) — yaml/build/at/ci:", auto_counts)

if __name__ == "__main__":
    main()


Indexing config files...
Indexing test files...
Done. Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\manual_review_sheet_autofilled.csv
Auto-detected counts (1's) — yaml/build/at/ci: {'yaml': 9, 'build': 134, 'at': 92, 'ci': 7}
